# Voir un gain : une question de fréquence d'observation · *Seeing a gain: a matter of observation frequency*

Notebook compagnon du chapitre **7. Module 1 en pratique : se bâtir une routine de veille macro** — [lire l'article](https://nmlab.io/ressources/se-batir-une-routine-de-veille-macro).
Companion notebook to chapter **7. Module 1 in Practice: Building Your Macro Monitoring Routine** — [read the article](https://nmlab.io/en/ressources/building-your-macro-monitoring-routine).

**Exécutez l'unique cellule ci-dessous** (bouton ▶ ou Ctrl+Entrée) : la figure est régénérée par le code — un **schéma éditable** : changez les libellés à votre guise. Passez `LANG = "en"` en tête de cellule pour les libellés anglais. — Run the single cell below (▶ or Ctrl+Enter) to rebuild the figure from code — an **editable diagram**: change the labels as you like; set `LANG = "en"` at the top for English labels.

Code : licence MIT · © 2026 [NMLab](https://nmlab.io) · dépôt [nmlab-finance/nmlab-figures](https://github.com/nmlab-finance/nmlab-figures)

In [ ]:
LANG = "fr"   # "fr" ou "en" — langue des libellés / label language

# Récupère puis active le style partagé NMLab (thème sombre + police Inter).
# Fetch and activate the shared NMLab style (dark theme + Inter font).
import urllib.request

urllib.request.urlretrieve("https://raw.githubusercontent.com/nmlab-finance/nmlab-figures/main/nmlab_style.py", "nmlab_style.py")
import nmlab_style as nm

nm.setup()


import math

def gain_probabilities() -> list[float]:
    """Probabilité (%) d'observer un gain selon la fréquence d'observation.

    Parabole du dentiste (Taleb, 2001) : un portefeuille d'espérance 15 %/an et de
    volatilité 10 %/an. Sur une fraction d'année ``t``, le rendement est un tirage normal
    de dérive ``0,15·t`` et d'écart-type ``0,10·√t`` ; la probabilité d'un gain vaut donc
    ``Φ(1,5·√t)`` (ratio de Sharpe = 0,15/0,10 = 1,5). Année de cotation : 252 séances de
    8 heures. / Normal-law probability of seeing a gain at each observation frequency."""
    sharpe = 0.15 / 0.10
    year = 252 * 8 * 3600                 # secondes de cotation par an (252 j × 8 h)
    periods = [1 / year, 60 / year, 3600 / year, 1 / 252, 1 / 12, 1 / 4, 1.0]
    phi = lambda x: 0.5 * (1 + math.erf(x / math.sqrt(2)))
    return [100 * phi(sharpe * math.sqrt(t)) for t in periods]


from matplotlib.figure import Figure

LABELS = {
    "fr": dict(
        title="Voir un gain : une question de fréquence d'observation",
        xlab="Fréquence d'observation du portefeuille",
        ylab="Probabilité d'observer un gain (%)",
        cats=["1 seconde", "1 minute", "1 heure", "1 jour", "1 mois", "1 trimestre", "1 an"],
        note="Probabilité qu'un coup d'œil affiche un gain — espérance 15 %/an, volatilité 10 %/an (Taleb, 2001).\n"
             "Ligne rouge : 50 % = pile ou face. Loi normale ; « 1 jour » = une séance de bourse."),
    "en": dict(
        title="Seeing a gain: a matter of observation frequency",
        xlab="Portfolio observation frequency",
        ylab="Probability of observing a gain (%)",
        cats=["1 second", "1 minute", "1 hour", "1 day", "1 month", "1 quarter", "1 year"],
        note="Probability that a glance shows a gain — 15%/yr expected return, 10%/yr volatility (Taleb, 2001).\n"
             "Red line: 50% = coin flip. Normal law; « 1 day » = one trading session."),
}

def _fmt(value: float, lang: str) -> str:
    """Étiquette de barre : 2 décimales sous 52 %, 1 au-dessus ; virgule + « % » en fr."""
    decimals = 2 if value < 52 else 1
    label = f"{value:.{decimals}f}"
    return label.replace(".", ",") + " %" if lang == "fr" else label + "%"

def build_figure(probabilities: list[float], lang: str) -> Figure:
    """Diagramme en barres : la compétence n'est visible qu'à basse fréquence."""
    text = LABELS[lang]
    fig = nm.figure(height_px=1235)
    ax = nm.axes(fig, top=1 - 168 / 1235, bottom=0.205)
    ax.grid(axis="x", visible=False)
    positions = range(len(probabilities))
    ax.bar(positions, probabilities, width=0.68, color=nm.COLORS["blue"], zorder=3)
    ax.axhline(50, color=nm.COLORS["rose"], linestyle="--", linewidth=2.6, zorder=2)
    ax.set_ylim(0, 103)
    ax.set_yticks(range(0, 101, 25))
    ax.set_ylabel(text["ylab"])
    ax.set_xlabel(text["xlab"], labelpad=14)
    ax.set_xlim(-0.7, len(probabilities) - 0.3)
    ax.set_xticks(list(positions))
    ax.set_xticklabels(text["cats"])
    for pos, value in zip(positions, probabilities):
        ax.annotate(_fmt(value, lang), (pos, value), xytext=(0, 12),
                    textcoords="offset points", ha="center", va="bottom",
                    fontsize=27, fontweight="bold", color=nm.COLORS["text"])
    nm.header(fig, text["title"])
    nm.footer(fig, text["note"])
    return fig

build_figure(gain_probabilities(), LANG)